# Online Retail Sales Analysis

Dataset : Online Retail (UK based gift store), Dec 2010 - Dec 2011

Things i want to find out:
- how are sales going month by month
- which products and countries make most money
- who are the best customers (RFM)
- how many customers come back (cohort)
- returns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import os
import json
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Loading the data

In [ ]:
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['LineTotal'] = df['Quantity'] * df['UnitPrice']
df.isnull().sum()

In [ ]:
df.describe()

Quantity and UnitPrice have negative / zero values, need to clean. Also CustomerID has lots of missing values (~25%) 

## 2. Cleaning

In [ ]:
# cancelled orders start with C
df['IsCancellation'] = df['InvoiceNo'].str.startswith('C', na=False) | (df['Quantity'] < 0)

df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0) & (df['IsCancellation'] == False)].copy()
df_returns = df[df['IsCancellation'] == True].copy()

df_clean['YearMonthStr'] = df_clean['InvoiceDate'].dt.strftime('%Y-%m')
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

print(len(df), len(df_clean), len(df_returns))

In [ ]:
gross_revenue = df_clean['LineTotal'].sum()
returns_value = abs(df_returns[df_returns['LineTotal'] < 0]['LineTotal'].sum())
net_revenue = gross_revenue - returns_value
total_orders = df_clean['InvoiceNo'].nunique()
total_units = df_clean['Quantity'].sum()
unique_products = df_clean['StockCode'].nunique()
unique_customers = df_clean['CustomerID'].nunique()
missing_cust_pct = df['CustomerID'].isnull().sum() / len(df) * 100
aov = gross_revenue / total_orders

print('Gross revenue:', gross_revenue)
print('Returns:', returns_value)
print('Net revenue:', net_revenue)
print('Orders:', total_orders)
print('Units:', total_units)
print('Products:', unique_products)
print('Customers:', unique_customers)
print('Missing customer %:', missing_cust_pct)
print('AOV:', aov)

## 3. Monthly sales

In [ ]:
monthly = df_clean.groupby('YearMonthStr').agg(
    Revenue=('LineTotal', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Units=('Quantity', 'sum')
).reset_index()
monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(monthly['YearMonthStr'], monthly['Revenue']/1000, color='blue')
ax1.set_ylabel('Revenue (k)')
ax1.set_xlabel('Month')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly['YearMonthStr'], monthly['Orders'], color='red', marker='o')
ax2.set_ylabel('Orders')

plt.title('Monthly Revenue and Orders (Dec 2010 - Dec 2011)')
plt.show()

Sales go up a lot from Sep to Nov, probably christmas season. Dec 2011 is low because data ends on 9 Dec.

## 4. Day and hour

In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']
day_hour = df_clean.groupby(['DayOfWeek', 'Hour'])['InvoiceNo'].nunique().unstack(fill_value=0)
day_hour = day_hour.reindex(days)

plt.figure(figsize=(12, 5))
sns.heatmap(day_hour, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Orders by Day and Hour (UTC)')
plt.show()

No orders on Saturday at all, weird. Most orders are around 12-3pm.

## 5. Top products

In [ ]:
top_prod = df_clean.groupby('Description').agg(
    Revenue=('LineTotal', 'sum'),
    Quantity=('Quantity', 'sum')
).sort_values(by='Revenue', ascending=False).head(10).reset_index()
top_prod

In [ ]:
top_prod = top_prod.sort_values(by='Revenue', ascending=True)

plt.figure(figsize=(11, 6))
plt.barh(top_prod['Description'], top_prod['Revenue']/1000)
plt.xlabel('Revenue (k)')
plt.title('Top 10 products by revenue')
plt.show()

Some of the top ones are not real products (like DOTCOM POSTAGE, Manual), should remove them maybe. Will do later

## 6. Country

In [ ]:
uk_val = df_clean[df_clean['Country'] == 'United Kingdom']['LineTotal'].sum()
intl_val = gross_revenue - uk_val
top_intl = df_clean[df_clean['Country'] != 'United Kingdom'].groupby('Country')['LineTotal'].sum().sort_values(ascending=False).head(8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.pie([uk_val, intl_val], labels=['UK (84.6%)', 'International'], autopct='%1.1f%%')
ax1.set_title('UK vs International')

ax2.barh(top_intl.index[::-1], top_intl.values[::-1]/1000)
ax2.set_title('Top 8 countries (without UK)')
ax2.set_xlabel('Revenue (k)')
plt.show()

## 7. RFM Segmentation

Recency = days since last order, Frequency = no. of orders, Monetary = total spend

In [ ]:
df_cust = df_clean[df_clean['CustomerID'].notnull()].copy()
ref_date = df_cust['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df_cust.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'LineTotal': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm = rfm[rfm['Monetary'] > 0]
rfm.head()

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent / New'
    elif r >= 3 and f <= 2:
        return 'Promising'
    elif r == 2 and f >= 3:
        return 'Needs Attention'
    elif r == 2 and f <= 2:
        return 'About To Sleep'
    elif r == 1 and f >= 4:
        return 'Cant Lose Them'
    elif r == 1 and f >= 2:
        return 'At Risk'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Segment'].value_counts()

In [ ]:
seg_agg = rfm.groupby('Segment').agg(
    CustomerCount=('CustomerID', 'count'),
    TotalRevenue=('Monetary', 'sum'),
    AvgRecency=('Recency', 'mean'),
    AvgFrequency=('Frequency', 'mean'),
    AvgMonetary=('Monetary', 'mean')
).reset_index()

seg_agg['CustomerShare'] = seg_agg['CustomerCount'] / len(rfm) * 100
seg_agg['SpendShare'] = seg_agg['TotalRevenue'] / rfm['Monetary'].sum() * 100
seg_agg = seg_agg.sort_values(by='TotalRevenue')
seg_agg

In [ ]:
y = np.arange(len(seg_agg))
plt.figure(figsize=(12, 6))
plt.barh(y + 0.2, seg_agg['SpendShare'], 0.4, label='Revenue share')
plt.barh(y - 0.2, seg_agg['CustomerShare'], 0.4, label='Customer share')
plt.yticks(y, seg_agg['Segment'])
plt.xlabel('%')
plt.title('RFM segments - customers vs revenue')
plt.legend()
plt.show()

Champions are small % of customers but give most of the revenue.

## 8. Pareto (80/20)

In [ ]:
sorted_m = rfm.sort_values(by='Monetary', ascending=False)
sorted_m['CumRevPct'] = sorted_m['Monetary'].cumsum() / sorted_m['Monetary'].sum() * 100
sorted_m['CustomerPct'] = np.arange(1, len(sorted_m)+1) / len(sorted_m) * 100

pct_cust_80 = sorted_m[sorted_m['CumRevPct'] >= 80]['CustomerPct'].iloc[0]
print(pct_cust_80)

plt.figure(figsize=(10, 6))
plt.plot(sorted_m['CustomerPct'], sorted_m['CumRevPct'])
plt.plot([0, 100], [0, 100], linestyle='--', color='grey')
plt.axhline(80, color='red', linestyle=':')
plt.axvline(pct_cust_80, color='red', linestyle=':')
plt.xlabel('% of customers')
plt.ylabel('% of revenue')
plt.title('Pareto - customer revenue')
plt.show()

## 9. Cohort retention

In [ ]:
df_cust['InvoiceMonth'] = df_cust['InvoiceDate'].dt.to_period('M')
df_cust['CohortMonth'] = df_cust.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

year_diff = df_cust['InvoiceMonth'].dt.year - df_cust['CohortMonth'].dt.year
month_diff = df_cust['InvoiceMonth'].dt.month - df_cust['CohortMonth'].dt.month
df_cust['CohortIndex'] = year_diff * 12 + month_diff + 1

cohort_data = df_cust.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0) * 100
retention.head()

In [ ]:
plt.figure(figsize=(14, 7))
sns.heatmap(retention, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=50)
plt.title('Cohort retention %')
plt.show()

Retention drops a lot after month 1. Dec 2010 cohort is the best one (maybe because its the biggest?)

## 10. Returns

In [ ]:
ret_monthly = df[df['LineTotal'] < 0].copy()
ret_monthly['YearMonthStr'] = ret_monthly['InvoiceDate'].dt.strftime('%Y-%m')
ret_by_month = ret_monthly.groupby('YearMonthStr')['LineTotal'].sum().abs() / 1000

plt.figure(figsize=(12, 5))
plt.bar(ret_by_month.index, ret_by_month.values, color='red')
plt.xticks(rotation=45)
plt.title('Returns per month (k)')
plt.show()

## 11. Save files

In [ ]:
os.makedirs('reports', exist_ok=True)

rfm.to_csv('reports/rfm_customer_segments.csv', index=False)
monthly.to_csv('reports/monthly_performance.csv', index=False)

country_summary = df_clean.groupby('Country').agg(
    Revenue=('LineTotal', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).sort_values(by='Revenue', ascending=False).reset_index()
country_summary.to_csv('reports/country_performance.csv', index=False)

top_products_all = df_clean.groupby(['StockCode', 'Description']).agg(
    Revenue=('LineTotal', 'sum'),
    UnitsSold=('Quantity', 'sum'),
    AveragePrice=('UnitPrice', 'mean'),
    OrderCount=('InvoiceNo', 'nunique')
).sort_values(by='Revenue', ascending=False).reset_index()
top_products_all.head(100).to_csv('reports/top_100_products.csv', index=False)

print('done')

## Conclusion

- Revenue peaks in Nov, so stock should be increased before that
- UK is ~85% of revenue, other countries are small
- top 20% customers gives around 80% of revenue (pareto)
- Many customers only buy once, need to do something to bring them back (email / discount)
- returns should be checked more, some months are very high

Next step: remove the postage/manual items and check the outliers